# 02. Baseline Benchmarks (LSTM · TCN · Transformer)

TGN과 같은 데이터·타깃·rolling window 설정으로 다변량 시계열 모델 3종을 평가합니다.


In [ ]:
!pip install torch_geometric
!pip install pyinform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 3.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import time
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ==========================================
# 0. 시드(Seed) 고정
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. 환경 설정 및 하이퍼파라미터
# ==========================================
time_window = 30
rolling_window = 60
epochs = 50
target_col_idx = 0
num_classes = 12
hidden_dim = 128        # 은닉층 크기

investable_assets = ['gold', 'vnq', 'xlk', 'xlf', 'xlv', 'xle', 'xly']

train_start = '2010-04-01'
test_end = '2017-12-31'

roll_train_years = 4
roll_val_years = 1
roll_test_years = 1
roll_step_years = 1

# ==========================================
# 2. 벤치마크 딥러닝 모델 정의 (LSTM, TCN, Transformer)
# ==========================================
class MultivariateLSTM(nn.Module):
    def __init__(self, num_assets, hidden_dim, num_classes=12):
        super().__init__()
        self.lstm = nn.LSTM(input_size=num_assets, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_assets * num_classes)
        self.num_assets = num_assets
        self.num_classes = num_classes

    def forward(self, x):
        out, _ = self.lstm(x)
        last_step = out[:, -1, :]
        logits = self.fc(last_step)
        return logits.view(self.num_assets, self.num_classes)

class MultivariateTCN(nn.Module):
    def __init__(self, num_assets, hidden_dim, num_classes=12):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=num_assets, out_channels=hidden_dim, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden_dim, num_assets * num_classes)
        self.num_assets = num_assets
        self.num_classes = num_classes

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out = self.relu(self.conv1(x))
        out = self.relu(self.conv2(out))
        last_step = out[:, :, -1]
        logits = self.fc(last_step)
        return logits.view(self.num_assets, self.num_classes)

class MultivariateTransformer(nn.Module):
    def __init__(self, num_assets, hidden_dim, num_classes=12, num_heads=4, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(num_assets, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_dim, num_assets * num_classes)
        self.num_assets = num_assets
        self.num_classes = num_classes

    def forward(self, x):
        x_proj = self.input_proj(x)
        out = self.transformer(x_proj)
        last_step = out[:, -1, :]
        logits = self.fc(last_step)
        return logits.view(self.num_assets, self.num_classes)

# ==========================================
# 3. 데이터 로더 및 전처리
# ==========================================
def load_csv_data(data_folder='./'):
    csv_files = glob.glob(os.path.join(data_folder, '**', '*.csv'), recursive=True)
    raw_data_dict = {}
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        asset_name = os.path.splitext(file_name)[0]
        if asset_name.lower() == 'manifest': continue
        try:
            df = pd.read_csv(file_path)
            df.columns = df.columns.str.strip()
            rename_dict = {col: 'Date' for col in df.columns if col.lower() in ['date', '날짜', 'time']}
            df.rename(columns=rename_dict, inplace=True)

            # 🔥 에러의 원인이었던 부분: 정상 파일만 필터링하는 안전장치 복구!
            if 'Date' in df.columns:
                raw_data_dict[asset_name] = df

        except Exception as e:
            print(f"[{asset_name}] 파일 로드 에러: {e}")
    return raw_data_dict

def preprocess_data(raw_data_dict):
    if 'vkospi' in raw_data_dict: del raw_data_dict['vkospi']
    asset_names = list(raw_data_dict.keys())
    processed_data = {}
    for name in asset_names:
        df = raw_data_dict[name].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').set_index('Date')
        target_col = df.columns[target_col_idx]

        if df[target_col].min() >= 0 and df[target_col].mean() > 1.0:
            df['Log_Return'] = np.log(df[target_col] / df[target_col].shift(1))
        else:
            df['Log_Return'] = df[target_col]

        df['rolling_mean'] = df['Log_Return'].rolling(window=rolling_window).mean()
        df['rolling_std'] = df['Log_Return'].rolling(window=rolling_window).std()
        df['scaled_value'] = (df['Log_Return'] - df['rolling_mean']) / (df['rolling_std'] + 1e-8)
        df = df.dropna()

        df['target_class'] = (np.floor(df['scaled_value']).clip(-6, 5) + 6).astype(int)
        df = df[(df.index >= train_start) & (df.index <= test_end)]
        processed_data[name] = df

    all_dates = set()
    for df in processed_data.values(): all_dates.update(df.index)
    global_calendar = pd.Series(list(all_dates)).sort_values().reset_index(drop=True)
    return global_calendar, processed_data, asset_names

# ==========================================
# 4. 모델용 3D 텐서 사전 구성 (딱 한 번만 실행!)
# ==========================================
def precompute_lstm_inputs(global_calendar, processed_data, asset_names, time_window):
    print(f"⏳ 입력 데이터(3D Tensor) 통합 구성 중... (모든 모델에서 재활용됨)")
    start_time = time.time()
    cache = {}
    num_assets = len(asset_names)

    for t_idx in range(time_window, len(global_calendar) - 1):
        past_dates = global_calendar[t_idx - time_window : t_idx]
        x_frame = np.zeros((time_window, num_assets))

        for i, name in enumerate(asset_names):
            available = processed_data[name].reindex(past_dates)['scaled_value'].fillna(0).values
            x_frame[:, i] = available

        next_date = global_calendar[t_idx + 1]
        y_dict = {}
        active_nodes = []

        for i, name in enumerate(asset_names):
            if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                y_dict[i] = processed_data[name].loc[next_date, 'target_class']
                active_nodes.append(i)

        cache[t_idx] = (torch.tensor(x_frame, dtype=torch.float).unsqueeze(0), y_dict, active_nodes)
        if t_idx % 1000 == 0: print(f"  - {t_idx}일 완료...")

    print(f"✅ 텐서 사전 구성 완료 (소요 시간: {time.time() - start_time:.2f}초)")
    return cache

# ==========================================
# 5. 메인 루프 (모델 순회 -> Core -> Others 연속 실행)
# ==========================================
def main():
    print(f"🚀 장치 설정: {device}")

    my_data_dir = '/content/drive/MyDrive/Find-A final'
    raw_data_dict = load_csv_data(my_data_dir)
    if len(raw_data_dict) == 0: return

    global_calendar, processed_data, asset_names = preprocess_data(raw_data_dict)
    num_asset = len(asset_names)
    criterion = nn.CrossEntropyLoss()

    print(f"📊 로드된 전체 자산 수: {num_asset}개")
    print(f"🗓️ 전체 가용 데이터 기간: {global_calendar.min().date()} ~ {global_calendar.max().date()}\n")

    lstm_data_cache = precompute_lstm_inputs(global_calendar, processed_data, asset_names, time_window)

    overall_start_date = global_calendar.min()
    overall_max_date = global_calendar.max()

    models_to_test = ['LSTM', 'TCN', 'Transformer']
    benchmark_modes = ['core', 'others']

    all_final_reports = {m: {} for m in models_to_test}

    for model_name in models_to_test:
        print("\n" + "="*50)
        print(f"🤖 [모델 변경] 아키텍처: {model_name}")
        print("="*50)

        for mode in benchmark_modes:
            print("\n" + "🔥"*30)
            print(f"🚀 [벤치마크 시작] 타겟 모드: {mode.upper()}")
            print("🔥"*30)

            step = 0
            final_test_results = []

            while True:
                t_start = overall_start_date + pd.DateOffset(years=(step * roll_step_years))
                t_end   = t_start + pd.DateOffset(years=roll_train_years) - pd.Timedelta(days=1)
                v_end   = t_end + pd.DateOffset(years=roll_val_years)
                te_end  = v_end + pd.DateOffset(years=roll_test_years)

                if te_end > overall_max_date: te_end = overall_max_date
                if v_end >= overall_max_date: break

                idx_t_start = global_calendar[global_calendar >= t_start].index[0]
                idx_t_start = max(time_window, idx_t_start)
                idx_t_end   = global_calendar[global_calendar <= t_end].index[-1]
                idx_v_end   = global_calendar[global_calendar <= v_end].index[-1]
                idx_te_end  = global_calendar[global_calendar <= te_end].index[-1]

                print(f"\n🔄 [{model_name} Step {step+1}] {global_calendar[idx_t_start].date()} ~ {global_calendar[idx_te_end].date()}")

                seed_everything(42 + step)

                if model_name == 'LSTM':
                    model = MultivariateLSTM(num_asset, hidden_dim, num_classes).to(device)
                elif model_name == 'TCN':
                    model = MultivariateTCN(num_asset, hidden_dim, num_classes).to(device)
                elif model_name == 'Transformer':
                    model = MultivariateTransformer(num_asset, hidden_dim, num_classes).to(device)

                optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005)

                best_val_acc, best_model_state = 0.0, None

                for epoch in range(epochs):
                    # --- TRAIN ---
                    model.train()
                    train_loss, train_cnt = 0, 0
                    for t_idx in range(idx_t_start, idx_t_end):
                        x_tensor, y_dict, active_nodes = lstm_data_cache.get(t_idx, (None, {}, []))
                        if x_tensor is None or len(active_nodes) == 0: continue

                        targets, valid_nodes = [], []
                        for node in active_nodes:
                            is_core = asset_names[node].lower() in [a.lower() for a in investable_assets]
                            if (mode == 'core' and is_core) or (mode == 'others' and not is_core):
                                targets.append(y_dict[node])
                                valid_nodes.append(node)

                        if not valid_nodes: continue

                        optimizer.zero_grad()
                        logits = model(x_tensor.to(device))
                        loss = criterion(logits[valid_nodes], torch.tensor(targets).to(device))
                        loss.backward()
                        optimizer.step()
                        train_loss += loss.item(); train_cnt += 1

                    # --- VAL ---
                    model.eval()
                    val_correct, val_dir, val_total = 0, 0, 0
                    with torch.no_grad():
                        for t_idx in range(idx_t_end, idx_v_end):
                            x_tensor, y_dict, active_nodes = lstm_data_cache.get(t_idx, (None, {}, []))
                            if x_tensor is None or len(active_nodes) == 0: continue

                            targets, valid_nodes = [], []
                            for node in active_nodes:
                                is_core = asset_names[node].lower() in [a.lower() for a in investable_assets]
                                if (mode == 'core' and is_core) or (mode == 'others' and not is_core):
                                    targets.append(y_dict[node]); valid_nodes.append(node)

                            if not valid_nodes: continue

                            y_true = torch.tensor(targets).to(device)
                            preds = model(x_tensor.to(device))[valid_nodes].argmax(dim=1)
                            val_correct += (preds == y_true).sum().item()
                            val_dir += ((preds >= 6) == (y_true >= 6)).sum().item()
                            val_total += len(targets)

                    val_acc = (val_correct / val_total) * 100 if val_total > 0 else 0
                    val_dir_acc = (val_dir / val_total) * 100 if val_total > 0 else 0
                    avg_train_loss = train_loss / train_cnt if train_cnt > 0 else 0

                    best_mark = "🌟" if val_acc > best_val_acc else ""
                    if val_acc > best_val_acc:
                        best_val_acc = val_acc
                        best_model_state = copy.deepcopy(model.state_dict())

                    if (epoch+1) % 10 == 0 or (epoch+1) == epochs:
                        print(f"  Epoch {epoch+1:02d}/{epochs} | Loss: {avg_train_loss:.4f} | Val Exact: {val_acc:.2f}% | Val Dir: {val_dir_acc:.2f}% {best_mark}")

                # --- TEST ---
                if best_model_state is not None:
                    model.load_state_dict(best_model_state)
                model.eval()
                t_correct, t_dir, t_total = 0, 0, 0
                with torch.no_grad():
                    for t_idx in range(idx_v_end, idx_te_end):
                        x_tensor, y_dict, active_nodes = lstm_data_cache.get(t_idx, (None, {}, []))
                        if x_tensor is None or len(active_nodes) == 0: continue

                        targets, valid_nodes = [], []
                        for node in active_nodes:
                            is_core = asset_names[node].lower() in [a.lower() for a in investable_assets]
                            if (mode == 'core' and is_core) or (mode == 'others' and not is_core):
                                targets.append(y_dict[node]); valid_nodes.append(node)

                        if not valid_nodes: continue

                        y_true = torch.tensor(targets).to(device)
                        preds = model(x_tensor.to(device))[valid_nodes].argmax(dim=1)

                        t_correct += (preds == y_true).sum().item()
                        t_dir += ((preds >= 6) == (y_true >= 6)).sum().item()
                        t_total += len(targets)

                res_exact = (t_correct/t_total)*100 if t_total > 0 else 0
                res_dir = (t_dir/t_total)*100 if t_total > 0 else 0
                print(f"🏁 Step {step+1} Test 👉 완벽 적중률: {res_exact:.2f}% | 방향 적중률: {res_dir:.2f}%")
                final_test_results.append((res_exact, res_dir))
                step += 1

            all_final_reports[model_name][mode] = final_test_results

    # ==========================================
    # 전체 벤치마크 결과 통합 리포트
    # ==========================================
    print("\n" + "✨"*30)
    print("📈 다중 모델 Baseline 통합 벤치마크 결과 📈")
    for model_name in models_to_test:
        print(f"\n[{model_name} 모델]")
        for mode in benchmark_modes:
            results = all_final_reports[model_name][mode]
            print(f" └─ {mode.upper()} 모드")
            for s, (e, d) in enumerate(results):
                print(f"    - Step {s+1}: 완벽 {e:.2f}% | 방향 {d:.2f}%")

            if len(results) > 0:
                avg_e = sum(x[0] for x in results) / len(results)
                avg_d = sum(x[1] for x in results) / len(results)
                print(f"    🏆 평균 👉 완벽: {avg_e:.2f}% | 방향: {avg_d:.2f}%")
    print("✨"*30)

if __name__ == "__main__":
    main()

🚀 장치 설정: cuda
📊 로드된 전체 자산 수: 32개
🗓️ 전체 가용 데이터 기간: 2010-04-01 ~ 2017-12-29

⏳ 입력 데이터(3D Tensor) 통합 구성 중... (모든 모델에서 재활용됨)
  - 1000일 완료...
  - 2000일 완료...
✅ 텐서 사전 구성 완료 (소요 시간: 31.21초)

🤖 [모델 변경] 아키텍처: LSTM

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🚀 [벤치마크 시작] 타겟 모드: CORE
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

🔄 [LSTM Step 1] 2010-05-13 ~ 2016-03-31
  Epoch 10/50 | Loss: 1.2770 | Val Exact: 33.67% | Val Dir: 49.77% 
  Epoch 20/50 | Loss: 0.8132 | Val Exact: 32.14% | Val Dir: 47.34% 
  Epoch 30/50 | Loss: 0.4705 | Val Exact: 30.67% | Val Dir: 46.94% 
  Epoch 40/50 | Loss: 0.2380 | Val Exact: 31.63% | Val Dir: 48.53% 
  Epoch 50/50 | Loss: 0.1123 | Val Exact: 31.52% | Val Dir: 48.92% 
🏁 Step 1 Test 👉 완벽 적중률: 34.47% | 방향 적중률: 50.11%

🔄 [LSTM Step 2] 2011-04-01 ~ 2017-03-31
  Epoch 10/50 | Loss: 1.2654 | Val Exact: 34.30% | Val Dir: 52.66% 
  Epoch 20/50 | Loss: 0.8367 | Val Exact: 32.54% | Val Dir: 50.68% 
  Epoch 30/50 | Loss: 0.4830 | Val Exact: 30.56% | Val Dir: 50.34% 
  Epoch 40/50 | Loss: 0.2348 | Val